# PATE: Private Aggregation of Teacher Ensembles

This notebook covers the current HTB subsection only: the PATE idea, its noisy vote aggregation formula, configuration, and a tiny reproducible example. Teacher training and student training belong to later subsections and are intentionally not implemented here.

## The architecture in plain language

PATE separates sensitive training from the deployed model. We split private data into disjoint partitions, train one **teacher** per partition, and ask all teachers to vote on an unlabeled public example. We add Laplace noise to the vote counts, publish only the noisy winning class, and train a **student** on those public examples and noisy labels.

The student never sees the private images or the teachers' full probability vectors. That creates an information bottleneck: private data is compressed into a small, noisy class label.

## Read the aggregation formula

\[\hat{y} = \operatorname{arg\,max}_{j}\left(n_j(x) + \operatorname{Lap}\left(\frac{1}{\epsilon}\right)\right)\]

**Say it aloud:** “y-hat equals arg-max over j of n sub j of x plus Laplace of one over epsilon.”

- **y-hat** (“y-hat”) is the released pseudo-label.
- **arg-max over j** (“argument maximum over jay”) means: return the class index `j` whose noisy score is largest.
- **n sub j of x** (“n sub jay of x”) is the number of teachers voting for class `j` on input `x`.
- **Laplace(1 over epsilon)** is random Laplace noise with scale `1/epsilon`. Smaller epsilon means a larger noise scale and stronger privacy.

In ordinary English: count each class's votes, perturb every count with independent noise, then choose the class with the largest noisy count.

In [ ]:
import numpy as np
import torch

RANDOM_SEED = 1337
NUM_CLASSES = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rng = np.random.default_rng(RANDOM_SEED)

print(f'Using device: {DEVICE}')

## Configuration from the HTB section

These values mirror the section's teaching setup. `num_teachers` and the model/training settings describe the later training phase; this notebook records them now so the privacy/utility trade-off is explicit.

In [ ]:
TEACHER_CONFIG = {
    'num_teachers': 250,
    'hidden_layers': [128, 64],
    'dropout': 0.2,
    'epochs': 30,
    'batch_size': 64,
    'learning_rate': 0.001,
}

AGGREGATION_CONFIG = {
    'noise_scale': 20.0,
    'num_student_queries': 5000,
}

STUDENT_CONFIG = {
    'hidden_layers': [128, 64],
    'dropout': 0.2,
    'epochs': 30,
    'batch_size': 64,
    'learning_rate': 0.001,
}

print(TEACHER_CONFIG)
print(AGGREGATION_CONFIG)
print(STUDENT_CONFIG)

In [ ]:
def noisy_aggregate(votes, num_classes, noise_scale, rng):
    """Implement y-hat = argmax_j(n_j(x) + Laplace(0, noise_scale))."""
    votes = np.asarray(votes, dtype=int)
    counts = np.bincount(votes, minlength=num_classes)
    noise = rng.laplace(loc=0.0, scale=noise_scale, size=num_classes)
    noisy_counts = counts + noise
    pseudo_label = int(np.argmax(noisy_counts))
    return pseudo_label, counts, noise, noisy_counts

# A strong consensus: 200 teachers vote for class 3 and 50 for class 7.
votes = np.array([3] * 200 + [7] * 50)
label, counts, noise, noisy_counts = noisy_aggregate(
    votes, NUM_CLASSES, AGGREGATION_CONFIG['noise_scale'], rng
)
print('Raw counts:', counts)
print('Noise:', np.round(noise, 2))
print('Noisy counts:', np.round(noisy_counts, 2))
print('Released pseudo-label:', label)

## Why consensus matters

The section uses noise scale 20. Since voting sensitivity is 2 (one record can move one vote from one class to another), the per-query privacy parameter is approximately **epsilon sub zero equals 2 divided by 20**, read aloud as “epsilon naught equals two over twenty,” or `0.10`.

A 200-to-50 vote gap is large compared with typical noise, so the winner usually survives. A 130-to-120 split has only a 10-vote gap, so noise can easily flip the result. That uncertainty is useful: it limits how much any single private record can determine.

For `m` queries, simply adding `m * epsilon_0` is a deliberately loose composition bound. The section's moments accountant gives a tighter cumulative bound; the exact accountant belongs to the later privacy-analysis work, so this notebook does not pretend the naive bound is the final guarantee.

## Section 4088: partition the data without leakage

We create three roles for the data: **private** data trains teachers, **public** data receives teacher labels later, and **holdout** data is used only for evaluation. The holdout set must stay untouched during training.

The scaler is fitted on private data only. In notation, if private features have mean \mu and standard deviation \sigma, each feature becomes x sub i prime equals (x sub i minus mu sub i) divided by sigma sub i. We then apply those same private-data parameters to public and holdout data.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from htb_ai_library import get_mnist_loaders

train_loader, test_loader = get_mnist_loaders()
X_train = train_loader.dataset.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_train = train_loader.dataset.targets.numpy()
X_test = test_loader.dataset.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_test = test_loader.dataset.targets.numpy()

X_private, X_public, y_private, y_public = train_test_split(
    X_train, y_train, test_size=0.20, random_state=RANDOM_SEED, stratify=y_train
)
X_holdout, y_holdout = X_test, y_test

scaler = StandardScaler()
X_private_norm = scaler.fit_transform(X_private)
X_public_norm = scaler.transform(X_public)
X_holdout_norm = scaler.transform(X_holdout)

print(f'Private: {len(X_private):,}; public: {len(X_public):,}; holdout: {len(X_holdout):,}')
print('Normalized feature shape:', X_private_norm.shape)

## Disjoint teacher partitions

We randomly permute private-data indices and give each teacher a separate slice. “Disjoint” means a private record belongs to exactly one teacher. That bounds how many teacher predictions can be influenced by one record. Shuffling first prevents accidental ordering patterns from becoming partition boundaries.

In [ ]:
num_teachers = TEACHER_CONFIG['num_teachers']
indices = rng.permutation(len(X_private_norm))
partition_size = len(X_private_norm) // num_teachers
teacher_partitions = []
for i in range(num_teachers):
    start = i * partition_size
    stop = len(X_private_norm) if i == num_teachers - 1 else start + partition_size
    teacher_partitions.append(indices[start:stop])

all_indices = np.concatenate(teacher_partitions)
assert len(all_indices) == len(X_private_norm)
assert len(np.unique(all_indices)) == len(X_private_norm)  # no overlap
print(f'{num_teachers} disjoint partitions; first size={len(teacher_partitions[0])}')

## Teacher training is intentionally guarded

The course trains 250 fresh MLPs. That is computationally expensive, so the notebook defines the exact loop but keeps execution behind `RUN_TEACHER_TRAINING = False`. Each teacher starts from new weights and sees only its own partition. Weight sharing would let information flow between partitions and weaken the privacy separation.

In [ ]:
from htb_ai_library import MLP, create_dataloader, train_model, get_model_predictions

RUN_TEACHER_TRAINING = False
teachers = []
if RUN_TEACHER_TRAINING:
    holdout_loader = create_dataloader(X_holdout_norm, y_holdout, 128, shuffle=False)
    for i, partition_idx in enumerate(teacher_partitions):
        teacher = MLP(input_size=784, hidden_layers=TEACHER_CONFIG['hidden_layers'],
                      num_classes=NUM_CLASSES, dropout=TEACHER_CONFIG['dropout'])
        X_teacher = X_private_norm[partition_idx]
        y_teacher = y_private[partition_idx]
        loader = create_dataloader(X_teacher, y_teacher, TEACHER_CONFIG['batch_size'])
        train_model(teacher, loader, holdout_loader, device=DEVICE,
                    epochs=TEACHER_CONFIG['epochs'],
                    learning_rate=TEACHER_CONFIG['learning_rate'])
        teachers.append(teacher)
    print('Trained teachers:', len(teachers))
else:
    print('Teacher training disabled; enable explicitly for the expensive 250-model run.')

## Raw ensemble votes

For each public example, every teacher predicts one class. The vote matrix has shape `(num_samples, num_classes)`. In mathematical notation, `votes[i, j]` means “the number of teachers that predicted class j for sample i.” This is the count matrix that the next PATE subsection will perturb with Laplace noise.

In [ ]:
def get_teacher_votes(teachers, X, device, num_classes=NUM_CLASSES):
    votes = np.zeros((len(X), num_classes), dtype=np.int32)
    for teacher in teachers:
        predictions = np.argmax(get_model_predictions(teacher, X, device), axis=1)
        for sample_index, predicted_class in enumerate(predictions):
            votes[sample_index, predicted_class] += 1
    return votes

if teachers:
    raw_votes = get_teacher_votes(teachers, X_public_norm, DEVICE)
    print('Raw vote matrix shape:', raw_votes.shape)
else:
    print('No teachers available yet; raw voting will run after training.')

## Section 4089: noisy aggregation and privacy accounting

The Laplace probability density is `f(z | b) = (1 / (2b)) exp(-|z| / b)`. Read aloud: “f of z given b equals one over two b times e to the negative absolute value of z over b.” Here `b` is the noise scale. A Laplace variable with scale 20 has standard deviation `b times square root of 2`, about 28.3.

For a vote histogram, one changed private record can move one vote between two classes, so sensitivity is 2. The per-query privacy cost is therefore `epsilon_0 = 2 / noise_scale`.

In [ ]:
def noisy_argmax(votes, noise_scale, rng=rng):
    """Add independent Laplace noise to every count and return winners."""
    votes = np.asarray(votes, dtype=np.float64)
    noise = rng.laplace(loc=0.0, scale=noise_scale, size=votes.shape)
    noisy_votes = votes + noise
    return np.argmax(noisy_votes, axis=1), noisy_votes

def compute_privacy_budget(num_queries, noise_scale, delta=1e-5):
    per_query_eps = 2.0 / noise_scale
    epsilon_sq_sum = num_queries * (per_query_eps ** 2)
    total_epsilon = np.sqrt(2 * epsilon_sq_sum * np.log(1 / delta))
    total_epsilon += num_queries * per_query_eps * (np.exp(per_query_eps) - 1)
    return total_epsilon, per_query_eps

total_eps, per_query_eps = compute_privacy_budget(5000, 20.0)
print(f'Per-query epsilon: {per_query_eps:.4f}')
print(f'Advanced-composition epsilon: {total_eps:.2f}')

## Pseudo-label generation

Each public sample consumes one privacy query. The ground-truth public labels may be retained for evaluation, but the student must receive only the noisy labels. The guarded cell below makes that distinction explicit and does not run until teachers exist.

In [ ]:
if teachers:
    num_queries = min(AGGREGATION_CONFIG['num_student_queries'], len(X_public_norm))
    query_indices = rng.choice(len(X_public_norm), num_queries, replace=False)
    X_query = X_public_norm[query_indices]
    y_query_true = y_public[query_indices]  # evaluation only; never sent to student
    votes = get_teacher_votes(teachers, X_query, DEVICE)
    student_labels, noisy_votes = noisy_argmax(votes, AGGREGATION_CONFIG['noise_scale'])
    clean_accuracy = (votes.argmax(axis=1) == y_query_true).mean()
    noisy_accuracy = (student_labels == y_query_true).mean()
    print(f'Clean ensemble accuracy: {clean_accuracy:.3f}')
    print(f'Noisy label accuracy: {noisy_accuracy:.3f}')
else:
    print('Pseudo-label generation deferred until the teacher ensemble is trained.')

## Section 4090: student training and deployment

PATE is hard-label knowledge distillation. The student receives a class index, not teacher probabilities. This reveals less about teacher confidence and therefore less about private training data. The privacy budget is spent while producing labels; later inference through the student does not access private data and consumes no additional PATE query budget.

In [ ]:
from safetensors.torch import save_file

RUN_STUDENT_TRAINING = False
student_model = None
if RUN_STUDENT_TRAINING and teachers:
    student_model = MLP(input_size=784, hidden_layers=STUDENT_CONFIG['hidden_layers'],
                        num_classes=NUM_CLASSES, dropout=STUDENT_CONFIG['dropout'])
    student_train_loader = create_dataloader(
        X_query, student_labels, STUDENT_CONFIG['batch_size']
    )
    train_model(student_model, student_train_loader, holdout_loader, device=DEVICE,
                epochs=STUDENT_CONFIG['epochs'],
                learning_rate=STUDENT_CONFIG['learning_rate'])
    student_test_acc = evaluate_accuracy(student_model, holdout_loader, DEVICE)
    print(f'Student holdout accuracy: {student_test_acc:.2f}%')
else:
    print('Student training disabled; enable after noisy pseudo-labels exist.')

### Why noisy labels can still train a useful model

Noise is concentrated on low-consensus examples. When teachers strongly agree, the vote margin is large and Laplace noise rarely changes the winner. Ambiguous examples receive more label flips, but stochastic gradient descent can tolerate moderate random label noise. The important distinction is that the noise is random rather than a systematic feature-based error.

In [ ]:
def deploy_model(model, new_data, scaler, device):
    """Run inference using private-fitted normalization; no private query occurs."""
    model.eval()
    X_norm = scaler.transform(new_data)
    X_tensor = torch.tensor(X_norm, dtype=torch.float32, device=device)
    with torch.no_grad():
        predictions = model(X_tensor).argmax(dim=1).cpu().numpy()
    return predictions

if student_model is not None:
    predictions = deploy_model(student_model, X_holdout[:100], scaler, DEVICE)
    print(f'Processed {len(predictions)} inference queries without new PATE budget.')
else:
    print('Deployment deferred until the student model is trained.')

In [ ]:
MODEL_DIR = 'models'
if student_model is not None:
    os.makedirs(MODEL_DIR, exist_ok=True)
    save_file(student_model.state_dict(), os.path.join(MODEL_DIR, 'pate_student.safetensors'))
    print('Saved models/pate_student.safetensors')
else:
    print('Model saving deferred; no trained student is present.')

## Section 4091: consensus as a quality signal

For each sample, `max_votes` is the largest class count. With 250 teachers, 200 votes means at least 80% agreement. High-consensus samples have a large margin, so Laplace noise is unlikely to change the winner. Low-consensus samples are ambiguous before noise is even added.

Confident aggregation labels only samples above a threshold. Rejected samples receive sentinel label `-1`; because MNIST classes are 0 through 9, `-1` cannot be mistaken for a real class.

In [ ]:
def confident_aggregation(votes, noise_scale, threshold, rng=rng):
    max_votes = votes.max(axis=1)
    confident_mask = max_votes >= threshold
    labels = np.full(len(votes), -1, dtype=np.int64)
    if confident_mask.any():
        confident_votes = votes[confident_mask].astype(np.float64)
        noise = rng.laplace(0.0, noise_scale, size=confident_votes.shape)
        labels[confident_mask] = np.argmax(confident_votes + noise, axis=1)
    return labels, confident_mask

def compute_confident_privacy_budget(num_queries, noise_scale, acceptance_rate, delta=1e-5):
    per_query_eps = 2.0 / noise_scale
    effective_queries = int(num_queries * acceptance_rate)
    epsilon_sq_sum = effective_queries * (per_query_eps ** 2)
    total_epsilon = np.sqrt(2 * epsilon_sq_sum * np.log(1 / delta))
    total_epsilon += effective_queries * per_query_eps * (np.exp(per_query_eps) - 1)
    return total_epsilon, per_query_eps, effective_queries

# A toy vote matrix makes the threshold behavior visible without training.
toy_votes = np.zeros((3, NUM_CLASSES), dtype=np.int32)
toy_votes[0, 3], toy_votes[0, 7] = 230, 20
toy_votes[1, 3], toy_votes[1, 7] = 180, 70
toy_votes[2, 3], toy_votes[2, 7] = 130, 120
labels, mask = confident_aggregation(toy_votes, 20.0, threshold=200)
print('Accepted mask:', mask)
print('Labels (-1 means rejected):', labels)
print('Budget:', compute_confident_privacy_budget(5000, 20.0, mask.mean()))

## Section 4096: tuning the privacy–utility trade-off

Increasing noise scale `b` decreases per-query epsilon because `epsilon_0 = 2 / b`, but it also makes label flips more likely. The useful operating point is usually an elbow: beyond it, privacy improves slowly while utility collapses quickly.

PATE is attractive when public data matching the private distribution exists and the deployed student will answer many queries. DP-SGD is preferable when no public proxy exists and the model must train directly on private data.

In [ ]:
NOISE_SCALES = [5.0, 10.0, 20.0, 40.0, 80.0]

def summarize_noise_tradeoff(votes, true_labels, num_queries, noise_scales=NOISE_SCALES):
    clean_labels = votes.argmax(axis=1)
    rows = []
    for scale in noise_scales:
        labels, _ = noisy_argmax(votes, scale)
        total_eps, per_eps = compute_privacy_budget(num_queries, scale)
        rows.append({
            'noise_scale': scale,
            'per_query_epsilon': per_eps,
            'total_epsilon': total_eps,
            'label_accuracy': float((labels == true_labels).mean()),
            'flip_rate': float((labels != clean_labels).mean()),
        })
    return rows

print('Noise scale increases -> stronger privacy, but usually lower label accuracy.')

In [ ]:
def estimate_queries_for_budget(target_epsilon, noise_scale, delta=1e-5):
    """Binary-search the largest query count within a budget."""
    per_query_eps = 2.0 / noise_scale
    low, high = 1, 100000
    while low < high:
        mid = (low + high + 1) // 2
        eps_sq_sum = mid * (per_query_eps ** 2)
        total_eps = np.sqrt(2 * eps_sq_sum * np.log(1 / delta))
        total_eps += mid * per_query_eps * (np.exp(per_query_eps) - 1)
        if total_eps <= target_epsilon:
            low = mid
        else:
            high = mid - 1
    return low

print('Queries within epsilon=10 at scale 20:', estimate_queries_for_budget(10.0, 20.0))

### Practical limitations

PATE assumes each person contributes to only one teacher partition. Duplicate or longitudinal records can violate that assumption. Public/private distribution shift can hurt more than random label noise, and class imbalance can give minority classes weaker consensus. These are data-governance constraints, not merely hyperparameter problems.

## Section 4093: membership-inference evaluation

A membership-inference attack asks whether a particular record was in a model's training set. We use maximum prediction confidence as the attack score. Members often receive unusually confident predictions because the model has fitted them directly.

Attack advantage is `attack accuracy minus 0.5`. Read aloud: “attack advantage equals attack accuracy minus one-half.” An advantage of 0.05 means 55% attack accuracy, five percentage points above random guessing.

In [ ]:
BASELINE_CONFIG = {
    'hidden_layers': [256, 128],
    'dropout': 0.0,
    'epochs': 50,
    'batch_size': 64,
    'learning_rate': 0.001,
    'train_size': 10000,
}

def compute_mia_advantage(model, X_members, X_nonmembers, device):
    member_conf = np.max(get_model_predictions(model, X_members, device), axis=1)
    nonmember_conf = np.max(get_model_predictions(model, X_nonmembers, device), axis=1)
    n = min(len(member_conf), len(nonmember_conf))
    scores = np.concatenate([member_conf[:n], nonmember_conf[:n]])
    labels = np.concatenate([np.ones(n), np.zeros(n)])
    thresholds = np.percentile(scores, np.linspace(0, 100, 1000))
    accuracies = [np.mean((scores >= threshold).astype(int) == labels) for threshold in thresholds]
    best_accuracy = float(max(accuracies))
    best_threshold = float(thresholds[int(np.argmax(accuracies))])
    return best_accuracy, best_accuracy - 0.5, best_threshold

print('MIA function defined; advantage is measured above the 50% random baseline.')

### Why the member/non-member sets must be balanced

If 90% of attack examples were members, an attacker could always guess “member” and achieve 90% accuracy without learning anything. Equal member and non-member counts make 50% the meaningful random baseline. The threshold search gives the attacker the best confidence cutoff, so this is a deliberately strong diagnostic.

In [ ]:
RUN_PRIVACY_EVALUATION = False
if RUN_PRIVACY_EVALUATION and student_model is not None:
    private_acc, private_adv, _ = compute_mia_advantage(
        student_model, X_private_norm, X_holdout_norm, DEVICE
    )
    print(f'PATE student attack accuracy: {private_acc:.4f}')
    print(f'PATE student private-data advantage: {private_adv:.4f}')
else:
    print('Privacy evaluation disabled; enable after a student model exists.')